# Symmetric Parseval CNN — denoising experiments (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Theborna/symmetric_parseval_conv/blob/main/colab_experiments.ipynb)

This notebook trains the four core models (`baseline`, `symmetric`, `mirror`,
`symmetric_mirror`) at several noise levels and produces the paper's PSNR/SSIM
table (Markdown + LaTeX).

**Before you start:** enable a GPU via *Runtime → Change runtime type → GPU*,
and have your BSD500 `train.h5` / `test.h5` ready (e.g. on Google Drive).


## 1. Check the runtime


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4 is fine).')

## 2. Get the code


In [ ]:
import os

REPO_DIR = 'symmetric_parseval_conv'
if not os.path.isdir(REPO_DIR) and os.path.basename(os.getcwd()) != REPO_DIR:
    !git clone https://github.com/Theborna/symmetric_parseval_conv.git
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)
!git pull --ff-only
print('Working dir:', os.getcwd())

## 3. Install dependencies

Colab already ships `torch`, `torchvision` and `numpy`; we only add the lighter
packages the repo imports.


In [ ]:
!pip install -q tqdm tensorboard h5py einops scikit-image matplotlib piqa pytorch-ssim

# utils/utilities.py imports pytorch_ssim at module load; make sure it resolves.
try:
    import pytorch_ssim, piqa  # noqa: F401
    print('SSIM deps OK')
except Exception as e:
    print('installing pytorch_ssim from source:', e)
    !pip install -q git+https://github.com/Po-Hsun-Su/pytorch-ssim.git

## 4. Point to your BSD500 data

The configs default to a local path that does not exist on Colab. Mount Drive
(or upload the files), set the two paths below, and the next cell rewrites the
data paths in every config. You need the pre-built HDF5 files `train.h5` and
`test.h5` (the same ones you train with locally).


In [ ]:
# Optional: mount Google Drive if your .h5 files live there.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, json

# <-- EDIT these two paths to point at your files -->
TRAIN_H5 = '/content/drive/MyDrive/BSD500/train.h5'
VAL_H5   = '/content/drive/MyDrive/BSD500/test.h5'

assert os.path.exists(TRAIN_H5), f'train file not found: {TRAIN_H5}'
assert os.path.exists(VAL_H5),   f'val file not found: {VAL_H5}'

def patch_data_paths(train_h5, val_h5):
    for p in ['config.json'] + sorted(glob.glob('experiment_configs/*.json')):
        with open(p) as f:
            cfg = json.load(f)
        if 'training_options' in cfg:
            cfg['training_options']['train_data_file'] = train_h5
            cfg['training_options']['val_data_file'] = val_h5
            with open(p, 'w') as f:
                json.dump(cfg, f, indent=4)
            print('patched', p)

patch_data_paths(TRAIN_H5, VAL_H5)

## 5. Run the sweep

`EPOCHS` overrides every config (use `1` for a quick smoke test first). Leave
`CONFIGS` empty to run all four models, or set e.g. `--configs symmetric_mirror mirror`.
Each config's own `sigmas` (default `[5, 15, 25]`) are used.


In [ ]:
EPOCHS  = 1              # bump up (e.g. 10) for the real run
OUTPUT  = 'exps/paper'
CONFIGS = ''             # e.g. '--configs symmetric_mirror mirror'

!python experiments.py -d {device} --epochs {EPOCHS} -o {OUTPUT} {CONFIGS}

## 6. Results


In [ ]:
from IPython.display import Markdown, display
with open(os.path.join(OUTPUT, 'results.md')) as f:
    display(Markdown(f.read()))

In [ ]:
# Per-run view (best validation metrics + wall-clock minutes)
import pandas as pd
res = json.load(open(os.path.join(OUTPUT, 'results.json')))
rows = []
for name, d in res.items():
    for sigma, m in d.get('runs', {}).items():
        rows.append({'model': d.get('label', name), 'sigma': int(sigma),
                     'PSNR': round(m['best_psnr'], 2), 'SSIM': round(m['best_ssim'], 4),
                     'depth': m.get('depth'), 'width': m.get('width'),
                     'minutes': m.get('minutes')})
pd.DataFrame(rows).sort_values(['model', 'sigma']).reset_index(drop=True)

In [ ]:
# LaTeX table for the paper
print(open(os.path.join(OUTPUT, 'results.tex')).read())

## 7. (Optional) Save the results

Copy the outputs to Drive so they survive the Colab session, or download them.


In [ ]:
# from google.colab import files
# files.download(os.path.join(OUTPUT, 'results.tex'))
# files.download(os.path.join(OUTPUT, 'results.json'))